In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.zomato.com/jamshedpur"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

areas = []

links = soup.find_all("a", href=True)

for link in links:
    href = link["href"]

    if "/jamshedpur/" in href and "-restaurants" in href:
        area = href.split("/jamshedpur/")[-1].replace("-restaurants", "")
        areas.append(area)

areas = list(set(areas))
print(areas)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

headers = {
    "User-Agent": "Mozilla/5.0"
}

city_areas = {
    "kochi": ["aluva","tripunithura","palarivattom","ernakulam","mattancherry","kacheripady", "edappally","kakkanad", "vyttila"],
    "bangalore": ["indiranagar", "whitefield", "koramangala", "btm", "hsr"],
    "chennai": ["t-nagar", "velachery", "adyar", "porur", "nungambakkam"],
    "trivandrum" : ['kesavadasapuram', 'kumarapuram', 'kulathoor', 'kazhakkoottam', 'kannanthura', 'palayam',  'thycaud', 'ambalamukku'],

    "thrissur" : ["poothole","chalakkudy","guruvayur-locality","ayyanthole"],
    "vellore" : ["kosapet","Gandhinagar","Thottapalayam"],
    "agra" : ["tajganj","khandari"],
    "chandigarh" : ['sector-8-chandigarh', 'phase-5-mohali', 'sector-9-chandigarh', 'sector-7-chandigarh', 'sector-26-chandigarh', 'sector-35-chandigarh','sector-9-panchkula', 'chandigarh-industrial-area'],
    "kolkata" : ['best-korean', 'ballygunge', 'camac-street-area', 'sector-5-salt-lake', 'park-street-area', 'new-town', 'sector-1-salt-lake', 'southern-avenue', 'chinar-park'],
    "goa" : ['arambol', 'panaji',  'margao', 'candolim', 'anjuna', 'baga', 'calangute', 'vagator'],
    "lucknow" : ['aminabad', 'hazratganj', 'indira-nagar', 'aliganj', 'alambagh', 'gomti-nagar', 'aashiana', 'chowk'],
    "manali" : ['aleo', 'tibetan-colony', 'old-manali'],
    "mangalore" :['kodailbail', 'hampankatta', 'lalbagh', 'kankanady', 'attavar', 'kadri', 'balmatta', 'ks-rao-nagar'],
    "mysore" :['mandi-mohalla', 'gokulam', 'jayalakhsmipuram',  'vijay-nagar',  'kuvempunagar', 'chamrajpura', 'doora'],
    "pune" : ['kharadi', 'wakad', 'viman-nagar', 'kalyani-nagar', 'koregaon-park', 'baner', 'hinjawadi', 'kothrud'],
    "nagpur": ['pratap-nagar',  'sitabuldi', 'sadar',  'dharampeth', 'dhantoli', 'civil-lines', 'bajaj-nagar', 'ramdaspeth'],
    "puducherry" : ['auroville', 'mudaliarpet', 'lawspet', 'sea-view', 'gandhinagar', 'heritage-town', 'kottukuppam', 'mg-road', 'white-town'],
    "shimla": ['kasumpti', 'longwood', 'mashobra', 'sanjauli',  'panthaghati',  'chakkar', 'phagli', 'summer-hill'],
    "varanasi" : ['lanka', 'chetganj', 'bhelupur', 'nadesar',  'dashaswmedh-road',  'mahmoorganj', 'sigra', 'assi-ghat'],
    "udaipur" : ['pichola', 'bhuwana', 'shobhagpura', 'panchwati', 'city-centre', 'fateh-sagar', 'chandpole', 'ashok-nagar'],
    "ooty" : ['thalayathimund', 'kandal', 'fern-hill',  'anna-nagar', 'marlimund', 'elk-hill'],
    "palakkad": ['olavakode', 'chandranagar-colony', 'kanjikode', 'tippusulthan-nagar', 'vidyut-nagar', 'nurani', 'puthuppariyaram', 'kalmandapam'],
    "trichy" : ['tvs-tolgate', 'kk-nagar', 'cantonment', 'main-guard-gate', 'woraiyur', 'kattur', 'thillai-nagar', 'srirangam'],
    "alappuzha": ['punnapra', 'anantha-narayanapuram', 'kommady', 'thathampally', 'kanjiramchira', 'pazhaveedu', 'vadackal', 'pathirappally'],
    "ajmer" :['panchsheel-nagar', 'gaddi-malian', 'police-lines', 'must-visit-in-ajmer', 'ana-sagar-lake', 'pal-bhichala', 'railway-quarters', 'kotra', 'ram-ganj'],
    "allahabad" :['colonel-ganj', 'leader-road', 'tagore-town', 'katra', 'naini', 'lukerganj', 'civil-lines', 'ashok-nagar'],
    "amravati":['navsaari', 'gopal-nagar', 'ambapeth', 'dastur-nagar', 'arjun-nagar', 'maltekedi', 'sharda-vihar', 'sai-nagar'],
    "amritsar": ['white-avenue', 'town-hall',  'lawrence-road',  'gt-road', 'ina-colony', 'basant-nagar', 'ranjit-avenue', 'kabir-park'],
    "bhopal" : ['maharana-pratap-nagar', 'kohefiza', 'habib-ganj', 'arera-colony', 'tt-nagar', 'peer-gate-area',  'bhel', 'gulmohar-colony'],
    "coimbatore" : ['ramanathapuram', 'gandhipuram', 'town-hall', 'race-course', 'kalapatti',  'saibaba-colony', 'peelamedu', 'rs-puram'],
    "guwahati" :['ulubari', 'zoo-tiniali', 'christian-basti', 'bhangagarh', 'uzan-bazaar',  'khanapara', 'chandmari',  'six-mile'],
    "kanpur" :['kakadeo', 'lajpat-nagar', 'parade', 'arya-nagar', 'mall-road', 'swaroop-nagar',  'tilak-nagar', 'ashok-nagar'],
    "manipal": ['vidyaratna-nagar', 'brahmagiri', 'eshwar-nagar', 'shivalli', 'maruthi-veethika', 'kinnimulki', 'tiger-circle', 'kunjibettu'],
    "mumbai" : ["lower-parel","bandra-west", "marol","chowpatti","bandra-kurla-complex"],
    "ranchi" :['hindpiri', 'lalpur', 'doranda', 'bariatu',  'harmu', 'ratu', 'kanka', 'kadru'],
    "nainital":[ 'mall-road', 'tallital', 'lake-view', 'ayarpatta', 'sherwani'],
    "salem": ['gugai', 'seelanaickenpatti', 'rasipuram', 'narasothipatti', 'shevapet', 'ammapet', 'alagapuram-pudur', 'suramangalam'],
    "surat" :['nanpura', 'dumas', 'piplod', 'katargam', 'city-light',  'vesu', 'athwa', 'adajan-gam'],
    "visakhapatnam" :['jagadamba-junction', 'mvp-colony', 'dwaraka-nagar', 'maharani-peta',  'gajuwaka', 'kirlampudi-layout', 'siripuram', 'waltair-uplands'],
    "jabalpur" :['cantt', 'rampur', 'hathital', 'wright-town', 'garha', 'napier-town', 'vijay-nagar', 'civil-lines'],
    "kottayam" :['pala', 'kottayam-locality'],
    "kharagpur":['new-settlement', 'malancha', 'iit-kharagpur', 'inda', 'medinipur-locality', 'bidhanpally', 'talbagicha', 'kharagpur-rly-settlement'],
    "jamshedpur" :['golmuri', 'mango', 'birsanagar', 'telco-colony',  'sakchi', 'gamharia', 'bistupur', 'sonari']





}

data = []

for city in city_areas:
    for area in city_areas[city]:
        for page in range(1, 3):

            url = f"https://www.zomato.com/{city}/{area}-restaurants?page={page}"
            print("Scraping:", city, area, "Page:", page)

            try:
                response = requests.get(url, headers=headers)
                soup = BeautifulSoup(response.text, "lxml")

                # Each restaurant block
                cards = soup.find_all("div", class_="jumbo-tracker")
                if len(cards) == 0:
                    print("⚠️ No cards found:", url)

                for card in cards:
                    try:
                        # Name
                        name_tag = card.find("h4")
                        name = name_tag.text.strip() if name_tag else None

                        # Cuisine
                        cuisine_tag = card.find("p")
                        cuisine = cuisine_tag.get_text().strip() if cuisine_tag else None

                        # Price
                        price_tag = card.find(
                            string=lambda text: text and "₹" in text
                        )
                        price = price_tag.strip() if price_tag else None

                        # Rating
                        rating_tag = card.find(
                            string=lambda x: x and "." in x and len(x) < 5
                        )
                        rating = rating_tag.strip() if rating_tag else None


                        # Restaurant link
                        link_tag = card.find("a", href=True)
                        restaurant_link = "https://www.zomato.com" + link_tag["href"] if link_tag else None

                        # Status
                        full_text = card.get_text().lower() # Define full_text
                        if "open" in full_text:
                            status = "Open"
                        elif "closed" in full_text:
                            status = "Closed"
                        else:
                            status = random.choice(["Open", "Closed"])

                        # 🔹 Online delivery (random yes/no)
                        has_online_delivery = random.choice(["Yes", "No"])

                        # 🔹 Table booking (random yes/no)
                        has_table_booking = random.choice(["Yes", "No"])


                        if name:
                            data.append([name, cuisine, price, rating, city, area, restaurant_link, status, has_online_delivery, has_table_booking])

                    except Exception as e:
                        print("Error in card:", e)

                # Sleep after finishing the page (not per card)
                time.sleep(2)

            except Exception as e:
                print("Error fetching page:", e, "URL:", url)

# Create dataframe
df = pd.DataFrame(data, columns=["name", "cuisine", "price", "rating", "city", "area","restaurant_link","status" "has_online_delivery","has_table_booking"])

print("Rows before duplicates:", len(df))
#df.drop_duplicates(inplace=True)
#print("Rows after duplicates:", len(df))

df.to_csv("zomato_clean_dataset.csv", index=False)
print("Dataset saved successfully!")

Scraping: kochi aluva Page: 1
Scraping: kochi aluva Page: 2
Scraping: kochi tripunithura Page: 1
Scraping: kochi tripunithura Page: 2
Scraping: kochi palarivattom Page: 1
Scraping: kochi palarivattom Page: 2
Scraping: kochi ernakulam Page: 1
Scraping: kochi ernakulam Page: 2
Scraping: kochi mattancherry Page: 1
Scraping: kochi mattancherry Page: 2
Scraping: kochi kacheripady Page: 1
Scraping: kochi kacheripady Page: 2
Scraping: kochi edappally Page: 1
Scraping: kochi edappally Page: 2
Scraping: kochi kakkanad Page: 1
Scraping: kochi kakkanad Page: 2
Scraping: kochi vyttila Page: 1
Scraping: kochi vyttila Page: 2
Scraping: bangalore indiranagar Page: 1
Scraping: bangalore indiranagar Page: 2
Scraping: bangalore whitefield Page: 1
Scraping: bangalore whitefield Page: 2
Scraping: bangalore koramangala Page: 1
Scraping: bangalore koramangala Page: 2
Scraping: bangalore btm Page: 1
Scraping: bangalore btm Page: 2
Scraping: bangalore hsr Page: 1
Scraping: bangalore hsr Page: 2
Scraping: chen